# B2-023 — Session 3: The Diffusion Forward Process

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

A diffusion model learns to generate data by learning to undo noise.
This session builds the half that needs no learning at all: the **forward process**, a fixed chain that adds a little Gaussian noise at each of $T$ steps until the data is indistinguishable from pure noise.
It introduces the variance schedule $\beta_t$, $\alpha_t$, $\bar\alpha_t$ with this unit's storage convention, teaches the one new probability fact the unit needs (a sum of independent Gaussians is Gaussian with added variances), and uses it to collapse $t$ noise steps into one closed-form draw.
Practices after this session: p03, p09, p15.

## 1. Destroying data slowly: the idea of a diffusion model

A **denoising diffusion probabilistic model (DDPM)** has two processes on the same chain of random variables $x_0,x_1,\dots,x_T$, each of the data's shape:

* the **forward process** $q$ starts from a data row $x_0$ and adds a small amount of Gaussian noise at every step, producing $x_1,x_2,\dots,x_T$; it is fixed in advance and has no parameters;
* the **reverse process** $p_\theta$ starts from pure noise $x_T\sim N(0,I)$ and removes noise one step at a time; a neural network supplies each step (Session 4).

The forward chain is **Markov**: $x_t$ is produced from $x_{t-1}$ alone, with fresh noise that is independent of everything before it. Because each step adds only a little noise, each reverse step is a small correction, which is why a network can learn it one step at a time.

**Why bother with a forward process at all?** Training needs pairs "noisy input, what the noise was". The forward process manufactures unlimited such pairs from the training rows, at every noise level, with no labels. Section 5 shows how to jump straight to any noise level in one draw.

**This unit's setting.** The data are the 2-D `mixture2d` rows (four blobs), $T=50$ steps, and every noise tensor is drawn from a dedicated seeded `torch.Generator` and passed in as an argument, exactly as in B2-022's reparameterization contract.

**Checkpoint 1A.** Which of the two processes has learnable parameters?

**Checkpoint 1B.** In a Markov chain, what does $x_t$ depend on directly?

**Collected answers.** Only the reverse process $p_\theta$; the forward process $q$ is fixed by the schedule. Only on $x_{t-1}$ and the fresh noise of step $t$; earlier states influence $x_t$ only through $x_{t-1}$.

## 2. The variance schedule and its storage convention

A **variance schedule** is a list $\beta_1,\dots,\beta_T$ with $0<\beta_t<1$: $\beta_t$ is the variance of the noise added at step $t$. Two derived quantities appear everywhere:
$$\alpha_t=1-\beta_t,\qquad \bar\alpha_t=\prod_{s=1}^{t}\alpha_s .$$
$\bar\alpha_t$ shrinks as $t$ grows; Section 5 shows it is the fraction of the original signal's variance that survives to step $t$.

**Linear schedule.** This unit spaces the betas evenly from $\beta_1$ to $\beta_T$: $\beta_t=\beta_1+(t-1)\frac{\beta_T-\beta_1}{T-1}$, which is exactly `torch.linspace(beta_1, beta_T, T)` (that call includes **both** endpoints and returns $T$ values). The unit's schedule is $T=50$, $\beta_1=0.001$, $\beta_T=0.2$.

**Storage convention (pinned for every practice).** Schedules are float64 tensors of length $T+1$ whose index $t$ holds timestep $t$, so code reads like the mathematics: `alpha_bar[t]` is $\bar\alpha_t$. Index $0$ is padding with $\beta_0=0$, $\alpha_0=1$, and $\bar\alpha_0=1$ (the empty product: no noise has been added yet). That padding makes $\bar\alpha_{t-1}$ valid at $t=1$, which Session 4 needs. The pinned constructor `make_schedule(T, beta_1, beta_T)` returns a dict with keys `"beta"`, `"alpha"`, `"alpha_bar"`, and `"beta_tilde"` (the last one is introduced in Session 4), each of shape `(T+1,)`; implementing it is p09.

**Small worked example.** With $T=3$ and $\beta=(0.1,0.2,0.5)$ (not linear, just for arithmetic): $\alpha=(0.9,0.8,0.5)$ and $\bar\alpha=(0.9,\ 0.72,\ 0.36)$. Stored with padding: `alpha_bar = [1.0, 0.9, 0.72, 0.36]`.

**Checkpoint 2A.** For `make_schedule(5, 0.1, 0.5)`, list `beta[1]` through `beta[5]` and state `beta[0]`.

**Checkpoint 2B.** Why does the stored `alpha_bar[0]` equal $1$?

**Collected answers.** $0.1, 0.2, 0.3, 0.4, 0.5$ (both endpoints included, spacing $0.1$), and `beta[0]` $=0$. Because $\bar\alpha_0$ is the empty product (no step has been applied, all of the signal remains), and it lets formulas that need $\bar\alpha_{t-1}$ work at $t=1$.

## 3. One forward step by reparameterization

One forward step is the Gaussian
$$q(x_t\mid x_{t-1})=N\big(\sqrt{\alpha_t}\,x_{t-1},\ \beta_tI\big).$$
B2-022's `gaussian-reparameterization` turns this into a sampling formula with the randomness drawn outside:
$$x_t=\sqrt{\alpha_t}\,x_{t-1}+\sqrt{\beta_t}\,\varepsilon_t,\qquad \varepsilon_t\sim N(0,I)\ \text{independent of }x_0,\dots,x_{t-1}.$$
Note the square roots: $\beta_t$ is a **variance**, so the noise is scaled by the **standard deviation** $\sqrt{\beta_t}$.

**Why shrink by $\sqrt{\alpha_t}$?** Consider one coordinate with $\operatorname{Var}(x_{t-1})=v$. Book 1 gives $\operatorname{Var}(cX)=c^2\operatorname{Var}(X)$, and `variance-of-sums` with `independence` gives $\operatorname{Var}(X+Y)=\operatorname{Var}X+\operatorname{Var}Y$ for independent $X,Y$. So
$$\operatorname{Var}(x_t)=\alpha_t v+\beta_t .$$
If $v=1$, then $\operatorname{Var}(x_t)=\alpha_t+\beta_t=1$: the step **preserves unit variance** while trading signal for noise. Without the shrink factor the variance would grow by $\beta_t$ at every step and the chain would never settle near $N(0,I)$.

**Checkpoint 3A.** With $x_{t-1}=2$, $\beta_t=0.36$, and $\varepsilon_t=-0.5$, compute $x_t$.

**Checkpoint 3B.** If $\operatorname{Var}(x_{t-1})=4$ and $\beta_t=0.36$, what is $\operatorname{Var}(x_t)$?

**Collected answers.** $\sqrt{0.64}\cdot2+\sqrt{0.36}\cdot(-0.5)=1.6-0.3=1.3$. $0.64\cdot4+0.36=2.92$: the variance moves toward $1$ but does not reach it in one step.

## 4. A new tool: sums of independent Gaussians

Collapsing many forward steps into one needs a fact Book 1 did not state.

**Sum rule.** If $U\sim N(0,s^2)$ and $W\sim N(0,r^2)$ are **independent**, then
$$U+W\sim N(0,\ s^2+r^2).$$

* **Gaussianity** of the sum is **stated** here, not proved (the proof needs tools beyond this course).
* The **mean** is $E[U+W]=E[U]+E[W]=0$ by linearity of `expectation`.
* The **variance** follows from Book 1: `variance-of-sums` gives $\operatorname{Var}(U+W)=\operatorname{Var}U+\operatorname{Var}W+2\operatorname{Cov}(U,W)$, and `independence` makes the `covariance` term $0$. So the variance is $s^2+r^2$.

**Vector form.** For vectors with independent coordinates, $N(0,s^2I)+N(0,r^2I)=N(0,(s^2+r^2)I)$ when the two vectors are independent: apply the scalar rule to each coordinate. In particular, if $\varepsilon_1,\varepsilon_2\sim N(0,I)$ are independent, then for constants $a,b$
$$a\varepsilon_1+b\varepsilon_2\ \text{has the same distribution as}\ \sqrt{a^2+b^2}\,\varepsilon,\qquad\varepsilon\sim N(0,I).$$

**Independence is essential.** If both terms used the **same** $\varepsilon$, then $a\varepsilon+b\varepsilon=(a+b)\varepsilon$ has variance $(a+b)^2$, not $a^2+b^2$. The cell below checks the rule by seeded Monte Carlo (Book 1 `sampling-simulation`) and shows the dependent case failing.

**Checkpoint 4A.** What is the distribution of $0.6\varepsilon_1+0.8\varepsilon_2$ for independent standard normals?

**Checkpoint 4B.** What is the variance of $0.6\varepsilon+0.8\varepsilon$ when both terms use the same $\varepsilon$?

**Collected answers.** $N(0,\,0.36+0.64)=N(0,1)$. $(1.4)^2=1.96$, because the terms are perfectly correlated and the covariance term $2\cdot0.6\cdot0.8=0.96$ does not vanish.

In [ ]:
import torch

g = torch.Generator().manual_seed(20261015)
e1 = torch.randn(400_000, generator=g, dtype=torch.float64)
e2 = torch.randn(400_000, generator=g, dtype=torch.float64)
independent = 0.6 * e1 + 0.8 * e2
same_noise = 0.6 * e1 + 0.8 * e1
print(f"independent: mean {independent.mean().item():+.4f}  var {independent.var().item():.4f}  (rule: 1.0000)")
print(f"same noise : mean {same_noise.mean().item():+.4f}  var {same_noise.var().item():.4f}  (1.4^2 = 1.9600)")
assert abs(independent.var().item() - 1.0) < 0.01 and abs(same_noise.var().item() - 1.96) < 0.02

## 5. The closed form $q(x_t\mid x_0)$

**Two steps, worked.** Apply Section 3 twice and expand:
$$x_2=\sqrt{\alpha_2}\,x_1+\sqrt{\beta_2}\,\varepsilon_2=\sqrt{\alpha_2}\big(\sqrt{\alpha_1}\,x_0+\sqrt{\beta_1}\,\varepsilon_1\big)+\sqrt{\beta_2}\,\varepsilon_2=\sqrt{\alpha_1\alpha_2}\,x_0+\underbrace{\sqrt{\alpha_2\beta_1}\,\varepsilon_1+\sqrt{\beta_2}\,\varepsilon_2}_{\text{noise}} .$$
The two noises are independent, so by Section 4 the noise term is $N\big(0,(\alpha_2\beta_1+\beta_2)I\big)$, and
$$\alpha_2\beta_1+\beta_2=\alpha_2(1-\alpha_1)+(1-\alpha_2)=1-\alpha_1\alpha_2=1-\bar\alpha_2 .$$
So $x_2=\sqrt{\bar\alpha_2}\,x_0+\sqrt{1-\bar\alpha_2}\,\varepsilon$ with a single $\varepsilon\sim N(0,I)$ independent of $x_0$.

**The general result.** For every $t\ge1$,
$$q(x_t\mid x_0)=N\big(\sqrt{\bar\alpha_t}\,x_0,\ (1-\bar\alpha_t)I\big),\qquad x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon .$$
It also holds at $t=0$ with the padding $\bar\alpha_0=1$. Proving it for all $t$ by induction, with the role of independence stated at each step, is p15.

**Why it matters.** Training never runs the chain step by step: for each row it picks a timestep $t$, draws one $\varepsilon$, and computes $x_t$ directly. The pinned function is `q_sample(x0, t, eps, alpha_bar)`: `x0` and `eps` have shape `(B,d)`, `t` is an int64 tensor of shape `(B,)` with values in $1,\dots,T$ (one timestep per row), and `alpha_bar` is the stored `(T+1,)` schedule. Indexing gives `alpha_bar[t]` of shape `(B,)`; `.unsqueeze(1)` makes it `(B,1)` so it broadcasts across the $d$ coordinates (Book 1 `broadcasting`); and it is cast to `x0`'s dtype so a float32 batch stays float32. The **clean** rows enter only through the `x0` argument, which is why later practices watch that argument to prove that no held-out row is ever noised during training.

**Checkpoint 5A.** With $\bar\alpha_t=0.36$, $x_0=(1,-2)$, and $\varepsilon=(0.5,0.5)$, compute $x_t$.

**Checkpoint 5B.** In `q_sample`, why is `alpha_bar[t]` unsqueezed to shape `(B,1)`?

**Collected answers.** $0.6\cdot(1,-2)+0.8\cdot(0.5,0.5)=(1.0,-0.8)$. So that each row's scalar coefficient multiplies both of that row's coordinates; a `(B,)` tensor would try to broadcast against the last axis `d` instead and either fail or pair the wrong numbers.

## 6. Signal-to-noise and why $\bar\alpha_T\approx0$

In $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$ the signal has variance factor $\bar\alpha_t$ and the noise $1-\bar\alpha_t$. Their ratio is the **signal-to-noise ratio**
$$\mathrm{SNR}(t)=\frac{\bar\alpha_t}{1-\bar\alpha_t},$$
which decreases strictly in $t$ because each $\alpha_t<1$.

**The endpoint.** Sampling (Session 4) starts from $x_T\sim N(0,I)$, which is only legitimate if $q(x_T\mid x_0)\approx N(0,I)$ for **every** data row, that is, if $\bar\alpha_T\approx0$. For the unit's schedule, $\bar\alpha_{50}=0.004506$ and $\sqrt{\bar\alpha_{50}}=0.0671$: a data point at distance $\sqrt2$ from the origin keeps a mean offset of about $0.095$ against unit noise.

**A rule of thumb.** For small $\beta$, $\log(1-\beta)\approx-\beta$, so $\log\bar\alpha_T\approx-\sum_t\beta_t$. The unit's schedule has $\sum_t\beta_t=50\cdot\frac{0.001+0.2}{2}=5.025$, so $\bar\alpha_T\approx e^{-5.025}=0.0066$; the exact value is smaller, because $\log(1-\beta)<-\beta$. The rule shows that **$T$ and the betas must be chosen together**: the popular linear schedule from $10^{-4}$ to $0.02$ was designed for $T=1000$; with only $T=50$ steps it gives $\sum\beta\approx0.5$ and $\bar\alpha_T\approx0.6$, nowhere near noise.

**Scale of the data.** The argument also assumes the data are of roughly unit scale or smaller: a row of size $100$ would keep a mean of $0.0671\cdot100\approx6.7$ at $t=T$. Session 5 returns to this through the latent scaling factor.

**Checkpoint 6A.** What is $\mathrm{SNR}(t)$ when $\bar\alpha_t=0.5$?

**Checkpoint 6B.** Estimate $\bar\alpha_T$ for $T=50$ with a linear schedule from $10^{-4}$ to $0.02$. Is $x_T$ close to pure noise?

**Collected answers.** $1$: signal and noise variances are equal. $\sum\beta\approx50\cdot0.01005=0.5025$, so $\bar\alpha_T\approx e^{-0.50}\approx0.6$. No: most of the signal survives, so starting the sampler from $N(0,I)$ would be wrong.

## 7. Worked laboratory: an $\bar\alpha_t$ table and a Monte Carlo check

**Worked exam-style example 1 (a numeric schedule).** Take $T=4$ with the linear schedule from $\beta_1=0.05$ to $\beta_4=0.35$.

| $t$ | $\beta_t$ | $\alpha_t$ | $\bar\alpha_t$ | $\sqrt{\bar\alpha_t}$ | $\sqrt{1-\bar\alpha_t}$ | $\mathrm{SNR}(t)$ |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 0.05 | 0.95 | 0.95 | 0.974679 | 0.223607 | 19.000000 |
| 2 | 0.15 | 0.85 | 0.8075 | 0.898610 | 0.438748 | 4.194805 |
| 3 | 0.25 | 0.75 | 0.605625 | 0.778219 | 0.627993 | 1.535658 |
| 4 | 0.35 | 0.65 | 0.39365625 | 0.627420 | 0.778681 | 0.649230 |

Each $\bar\alpha_t$ is the previous one times $\alpha_t$: $0.95\cdot0.85=0.8075$, then $\cdot0.75=0.605625$, then $\cdot0.65=0.39365625$.

**Worked exam-style example 2 (reading the table).** *At which first timestep does noise variance exceed signal variance?* When $\mathrm{SNR}(t)<1$, that is $\bar\alpha_t<\tfrac12$: first at $t=4$. *Where does $x_0=(1,-1)$ go at $t=3$ with $\varepsilon=(0,0)$?* To $\sqrt{\bar\alpha_3}\,x_0=(0.778219,-0.778219)$, the conditional mean.

**Monte Carlo check.** The first cell builds this schedule with the storage convention, prints the table, and verifies the closed form by drawing $400{,}000$ values of $x_3$ from the fixed row $x_0=(1,-1)$: the sample mean is close to $\sqrt{\bar\alpha_3}\,x_0$ and the per-coordinate variance close to $1-\bar\alpha_3=0.394375$. The second cell confirms that running the chain step by step gives the same distribution as the one-draw closed form.

**Checkpoint 7A.** From the table, what is $\bar\alpha_2$, and what is $1-\bar\alpha_2$?

**Checkpoint 7B.** Compute $\mathrm{SNR}(4)$ from $\bar\alpha_4$ to four decimals.

**Collected answers.** $\bar\alpha_2=0.8075$ and $1-\bar\alpha_2=0.1925$. $0.39365625/0.60634375=0.6492$.

In [ ]:
import torch

T = 4
beta = torch.zeros(T + 1, dtype=torch.float64)
beta[1:] = torch.linspace(0.05, 0.35, T, dtype=torch.float64)    # index 0 is padding
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)                           # alpha_bar[0] = 1
for t in range(1, T + 1):
    snr = alpha_bar[t] / (1 - alpha_bar[t])
    print(f"t={t}: beta={beta[t].item():.2f} alpha={alpha[t].item():.2f} alpha_bar={alpha_bar[t].item():.8f} "
          f"sqrt(ab)={alpha_bar[t].sqrt().item():.6f} sqrt(1-ab)={(1 - alpha_bar[t]).sqrt().item():.6f} SNR={snr.item():.6f}")

g = torch.Generator().manual_seed(20261015)
x0 = torch.tensor([1.0, -1.0], dtype=torch.float64).expand(400_000, 2)
eps = torch.randn(400_000, 2, generator=g, dtype=torch.float64)
x3 = alpha_bar[3].sqrt() * x0 + (1 - alpha_bar[3]).sqrt() * eps
print("sample mean", x3.mean(dim=0).tolist(), " target", (alpha_bar[3].sqrt() * x0[0]).tolist())
print("sample var ", x3.var(dim=0).tolist(), " target", (1 - alpha_bar[3]).item())
assert torch.allclose(x3.mean(dim=0), alpha_bar[3].sqrt() * x0[0], atol=0.01, rtol=0.0)
assert torch.allclose(x3.var(dim=0), (1 - alpha_bar[3]).expand(2), atol=0.01, rtol=0.0)

In [ ]:
import torch

T = 4
beta = torch.zeros(T + 1, dtype=torch.float64)
beta[1:] = torch.linspace(0.05, 0.35, T, dtype=torch.float64)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)
g = torch.Generator().manual_seed(20261016)
x = torch.full((400_000,), 1.0, dtype=torch.float64)                # x0 = 1 for every draw
for t in range(1, T + 1):                                          # run the chain one step at a time
    x = alpha[t].sqrt() * x + beta[t].sqrt() * torch.randn(400_000, generator=g, dtype=torch.float64)
print(f"step by step: mean {x.mean().item():.4f}  var {x.var().item():.4f}")
print(f"closed form : mean {alpha_bar[T].sqrt().item():.4f}  var {(1 - alpha_bar[T]).item():.4f}")
assert abs(x.mean().item() - alpha_bar[T].sqrt().item()) < 0.01 and abs(x.var().item() - (1 - alpha_bar[T]).item()) < 0.01

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *$\alpha_t$ where $\bar\alpha_t$ belongs.* `alpha[t]` in `q_sample` produces one step of noise from $x_0$, not $t$ steps; the two agree only at $t=1$. Fix: `alpha_bar[t]`, and probe at some $t\ge2$.
2. *Variance used as standard deviation.* `(1 - alpha_bar[t]) * eps` instead of `(1 - alpha_bar[t]).sqrt() * eps`. Fix: square roots on both coefficients.
3. *Off-by-one schedules.* `torch.linspace(beta_1, beta_T, T + 1)`, or storing $\beta_1$ at index $0$, shifts every timestep. Fix: the pinned $(T+1)$ layout with padding at index $0$, and probe `beta[1] == beta_1` and `beta[T] == beta_T`.
4. *Reusing one noise draw.* Sums of the same $\varepsilon$ are not the sum rule's independent case. Fix: independent noise for independent steps; one $\varepsilon$ per row in the closed form.
5. *A schedule that does not reach noise.* Check $\bar\alpha_T$ before training; with $T=50$ the betas must be much larger than the $T=1000$ defaults.
6. *Broadcasting the coefficient on the wrong axis.* A `(B,)` coefficient against a `(B,d)` batch. Fix: `.unsqueeze(1)`.

**Exam connections.** p03 computes $\bar\alpha_t$ from a literal three-step schedule in multiple-choice register; p09 implements `make_schedule` and `q_sample` against exact probes (including a statistical probe of the closed form); p15 proves $q(x_t\mid x_0)=N(\sqrt{\bar\alpha_t}x_0,(1-\bar\alpha_t)I)$ by induction with the sum rule; p23 (after Session 5) repairs an implementation with an $\alpha$/$\bar\alpha$ confusion.

**Checkpoint 8A.** At which timestep do `q_sample` with `alpha` and with `alpha_bar` agree, and why does that make $t=1$ a useless probe?

**Checkpoint 8B.** Which practice proves the closed form for every $t$?

**Collected answers.** Only at $t=1$, because $\bar\alpha_1=\alpha_1$; a probe at $t=1$ cannot distinguish the bug from the correct code. p15, by induction.

## 9. Forward-only deeper topic: continuous time, scores, and other schedules

Forward-only deeper topic.
Letting the number of steps grow without bound turns the forward chain into a stochastic differential equation, and the reverse process can then be described through the **score** $\nabla_x\log q(x_t)$ of each noisy distribution; this "score-based" view unifies DDPMs with other generative models and allows samplers that take far fewer steps.
Other schedules (for example a cosine schedule for $\bar\alpha_t$) spread the signal-to-noise ratio more evenly across timesteps than the linear one.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-023: every practice uses the linear schedule and the discrete chain of this session.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Does any B2-023 practice use a cosine schedule?

**Checkpoint 9B.** Which quantity does the score-based view attach to each noise level?

**Collected answers.** No; every practice uses the pinned linear schedule. The score, $\nabla_x\log q(x_t)$; this is forward-only and not assessed.